**1. Imports & Hardware Configuration**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import zipfile
import os
drive.mount('/content/drive')

# 2. Define paths
# Read from your Google Drive
zip_path = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/augmented_data.zip"

# Extract to Colab's fast local SSD storage
extract_path = '/content/dataset'

# 3. Unzip the dataset into Colab's temporary fast storage
if not os.path.exists(extract_path):
    print(f"Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Unzipping complete!")
else:
    print("Dataset already extracted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already extracted.


**2. Data Loading**

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Point specifically to the training folder inside the extracted zip
data_dir = "/content/dataset/training"

transform = transforms.Compose([
    transforms.Resize((240, 320)), # RVC2 required dimensions
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = ImageFolder(root=data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

print(f"Loaded {len(dataset)} training images.")

Loaded 3928 training images.


**3. Initialize Teacher Model (DINOv2) and Student Model MobileNetV3-Small**

In [ ]:
# ==========================================
# TEACHER & STUDENT INITIALIZATION
# ==========================================
import torch
import torch.nn as nn
from transformers import AutoModel
import torchvision.models as models

# --- 1. DEFINE THE TEACHER ARCHITECTURE ---
class HikingModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Load the base DINOv2 model
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-small")
        self.embed_dim = 384

        # Freeze backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Custom MLP Head
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values)
        # Extract the CLS token
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_token)

# --- 2. LOAD THE TEACHER WEIGHTS ---
print("Initializing Teacher Model (DINOv2)...")
teacher_model = HikingModel()

# Upload your .pth file to Colab
checkpoint_path = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/path_rater_dinov2.pth"

# Safely load the dictionary
checkpoint = torch.load(checkpoint_path, map_location=device)
state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
teacher_model.load_state_dict(state_dict)

teacher_model.to(device)
teacher_model.eval() # Freeze for distillation
print("Teacher loaded successfully!")


# --- 3. INITIALIZE THE STUDENT MODEL ---
print("Initializing Student Model (MobileNetV3-Small)...")
# We start with a blank (untrained) edge-friendly model
student_model = models.mobilenet_v3_small(weights="DEFAULT")

# Modify the final classification layer to output a single score (like the teacher)
in_features = student_model.classifier[3].in_features
student_model.classifier[3] = nn.Linear(in_features, 1)

student_model.to(device)
student_model.train() # Set to training mode
print("Student initialized successfully!")

Initializing Teacher Model (DINOv2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Teacher loaded successfully!
Initializing Student Model (MobileNetV3-Small)...
Student initialized successfully!


**4. Distillation Setup**

In [ ]:
# MSE is the standard loss function for continuous scalar values
criterion = nn.MSELoss()
optimizer = optim.Adam(student_model.parameters(), lr=0.001)

**5. Training Loop**

In [ ]:
epochs = 10
for epoch in range(epochs):
    running_loss = 0.0
    for images, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
        images = images.to(device)

        # Forward pass teacher (no gradients needed)
        with torch.no_grad():
            teacher_logits = teacher_model(images)

        # Forward pass student
        student_logits = student_model(images)

        # Calculate loss and update student
        loss = criterion(student_logits, teacher_logits)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Average Loss: {running_loss/len(dataloader):.4f}")

Epoch 1/10: 100%|██████████| 123/123 [00:33<00:00,  3.68it/s]


Average Loss: 2.3654


Epoch 2/10: 100%|██████████| 123/123 [00:32<00:00,  3.78it/s]


Average Loss: 0.9862


Epoch 3/10: 100%|██████████| 123/123 [00:35<00:00,  3.47it/s]


...

Average Loss: 0.2747


Epoch 9/10: 100%|██████████| 123/123 [00:35<00:00,  3.47it/s]


Average Loss: 0.3106


Epoch 10/10: 100%|██████████| 123/123 [00:35<00:00,  3.45it/s]

Average Loss: 0.2434


**6. Export the Distilled Student to ONNX**

In [ ]:
!pip install onnx onnxscript

In [ ]:
# print("Exporting Student Model to ONNX...")

# # Freeze the layers for hardware inference
# student_model.eval()

# # Wrap the model so sigmoid is baked into the exported graph
# # This way the blob outputs a 0-1 scenic score directly, no post-processing needed on the Pi
# class StudentForExport(nn.Module):
#     def __init__(self, model):
#         super().__init__()
#         self.model = model
#     def forward(self, x):
#         return torch.sigmoid(self.model(x))

# export_model = StudentForExport(student_model).cpu().eval()

# # Save the trained weights before exporting (safety net in case you need to re-export later)
# torch.save(student_model.state_dict(),"/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3.pth")

# dummy_input = torch.randn(1, 3, 240, 320, device=device)
# torch.onnx.export(
#     export_model.cpu(),
#     dummy_input.cpu(),
#     "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3.onnx",
#     export_params=True,
#     opset_version=12,
#     do_constant_folding=True,
#     input_names=['input'],
#     output_names=['output'],
#     dynamo=False
# )
# print("Export complete")




Exporting Student Model to ONNX...


/tmp/ipykernel_73507/42350324.py:21: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export complete


In [ ]:
print("Exporting Student Model to ONNX...")

student_model.eval()

# === Replace HardSigmoid/HardSwish with Myriad-compatible equivalents ===
# The Myriad VPU doesn't support HardSigmoid as a standalone op.
# HardSigmoid(x) = clamp(x/6 + 0.5, 0, 1)  — we rewrite using ReLU6
# HardSwish(x)   = x * HardSigmoid(x)       — same rewrite

import torch.nn.functional as F

class MyriadHardSigmoid(nn.Module):
    def forward(self, x):
        return F.relu6(x + 3.0) / 6.0

class MyriadHardSwish(nn.Module):
    def forward(self, x):
        return x * (F.relu6(x + 3.0) / 6.0)

def replace_activations(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Hardswish):
            setattr(module, name, MyriadHardSwish())
        elif isinstance(child, nn.Hardsigmoid):
            setattr(module, name, MyriadHardSigmoid())
        else:
            replace_activations(child)

replace_activations(student_model)

# Wrap with sigmoid for 0-1 output
class StudentForExport(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return torch.sigmoid(self.model(x))

export_model = StudentForExport(student_model).cpu().eval()

# Save weights
torch.save(student_model.state_dict(),
           "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3.pth")

dummy_input = torch.randn(1, 3, 240, 320)

torch.onnx.export(
    export_model,
    dummy_input,
    "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3.onnx",
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamo=False,
)
print("Export complete")

Exporting Student Model to ONNX...


/tmp/ipykernel_73507/773072271.py:47: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export complete


**7. Evaluate Student Model on Testing Data**

In [ ]:
import torch
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

print("Evaluating Student Model on Testing Data...")

# 1. Move model back to the GPU (it was left on the CPU after ONNX export)
student_model.to(device)

# Ensure the model is locked in evaluation mode
student_model.eval()

# 2. Load the testing dataset
test_dir = "/content/dataset/testing"
test_dataset = ImageFolder(root=test_dir, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Map PyTorch's generated class indices back to your actual ground truth floats
idx_to_score = {i: float(cls_name) for i, cls_name in enumerate(test_dataset.classes)}
print(f"Ground Truth Mapping: {idx_to_score}")

total_mae = 0.0
total_mse = 0.0
total_samples = 0

# 4. Run the Evaluation Loop
with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)

        # Get Student Predictions and apply Sigmoid to bound between 0 and 1
        predictions = torch.sigmoid(student_model(images)).squeeze()

        # Get Ground Truths (convert the integer labels to your float scores)
        ground_truths = torch.tensor([idx_to_score[label.item()] for label in labels], device=device)

        # Handle edge case if the last batch only has 1 image
        if predictions.dim() == 0:
            predictions = predictions.unsqueeze(0)

        # Calculate errors
        absolute_errors = torch.abs(predictions - ground_truths)
        squared_errors = torch.square(predictions - ground_truths)

        total_mae += torch.sum(absolute_errors).item()
        total_mse += torch.sum(squared_errors).item()
        total_samples += labels.size(0)

average_mae = total_mae / total_samples
average_mse = total_mse / total_samples

print("-" * 30)
print(f"Tested on {total_samples} unseen images.")
print(f"Mean Absolute Error (MAE): {average_mae:.4f}")
print(f"Mean Squared Error (MSE):  {average_mse:.4f}")
print("-" * 30)

Evaluating Student Model on Testing Data...
Ground Truth Mapping: {0: 0.0, 1: 0.3, 2: 0.6, 3: 1.0}
------------------------------
Tested on 453 unseen images.
Mean Absolute Error (MAE): 0.1519
Mean Squared Error (MSE):  0.0455
------------------------------


**8. Convert ONNX Student Model to .blob File for Luxonis Camera Coompatibility**

First, we need to we need to run the .onnx file through the onnxsim (ONNX Simplifier) Python package before sending it to the blob converter. onnxsim strips the incompatible PyTorch metadata, mathematically simplifies the network operations, and repackages the file into a clean, standard ONNX format that OpenVINO accepts.

In [ ]:
# Install ONNX Simplifier
!pip install onnxsim

# Define file paths
input_onnx = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3.onnx"
simplified_onnx = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3_simplified.onnx"

# Run the simplifier
!onnxsim "{input_onnx}" "{simplified_onnx}"

  Using cached onnxsim-0.6.2-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (4.5 kB)
  Using cached onnx-1.21.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.5 kB)
Using cached onnxsim-0.6.2-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (2.9 MB)
Using cached onnx-1.21.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (17.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 154.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [onnxsim]
Installing onnxruntime by `/content/miniforge/bin/python3.13 -m pip install 
onnxruntime`, please wait for a moment..
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 147.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 144.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53

In [ ]:
# ==========================================
# CONVERT ONNX → OpenVINO IR → .BLOB FOR OAK-D LITE
# ==========================================
# Step 1: ONNX → IR using OpenVINO 2022.3 (needs Python 3.10, Colab has 3.12)
# Step 2: IR → blob via blobconverter cloud (has a physical Myriad device)
# ==========================================

import os, shutil

# --- Step 1: ONNX → OpenVINO IR (conda Python 3.10 env) ---
# Install Miniforge if not already present
if not os.path.exists("/content/miniforge"):
    os.system("wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -b -p /content/miniforge")

os.environ["PATH"] = "/content/miniforge/bin:" + os.environ["PATH"]

!conda create -y -n ov python=3.10 -q 2>/dev/null || true
OV_PY = "/content/miniforge/envs/ov/bin/python"
!{OV_PY} -m pip install -q openvino-dev==2022.3.0

onnx_path = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment/student_mobilenet_v3_simplified.onnx"
output_dir = "/content/drive/MyDrive/Spring 2026/AI ML Project Course/24-782 Team Third Eye/Phase 3/Edge Model Deployment"
ir_dir = "/content/ir_output"
os.makedirs(ir_dir, exist_ok=True)

!{OV_PY} -m openvino.tools.mo \
    --input_model "{onnx_path}" \
    --output_dir {ir_dir} \
    --model_name student_mobilenet_v3 \
    --compress_to_fp16 \
    --mean_values "[123.675,116.28,103.53]" \
    --scale_values "[58.395,57.12,57.375]" \
    --reverse_input_channels

# --- Step 2: IR → .blob via blobconverter cloud API ---
!pip install -q blobconverter
import blobconverter

blob_path = blobconverter.from_openvino(
    xml=f"{ir_dir}/student_mobilenet_v3.xml",
    bin=f"{ir_dir}/student_mobilenet_v3.bin",
    data_type="FP16",
    shaves=6,
    version="2022.1",
    use_cache=False,
)

shutil.copy(blob_path, f"{output_dir}/student_mobilenet_v3.blob")
print(f"\nDone! Blob saved to: {output_dir}/student_mobilenet_v3.blob")

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /content/miniforge/envs/ov

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.2.25-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.3-h33c6efd_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.7.5-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_18 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.0-h69a702a_18 
  libgomp            conda-forge/linux-64::libgomp-15.2.0-he0feb66_18 
  liblzma            conda-forg

## Deployment Pipeline: ONNX → OpenVINO IR → MyriadX Blob

### Why this is a two-step process

The OAK-D Lite runs on Intel's MyriadX VPU (RVC2 platform), which requires models
in `.blob` format. Getting from a PyTorch model to a `.blob` involves:

1. **PyTorch → ONNX** (done in the export cell above)
2. **ONNX → OpenVINO IR** (`.xml` + `.bin`) using OpenVINO's Model Optimizer
3. **OpenVINO IR → `.blob`** using OpenVINO's compile_tool (requires a Myriad device)

### Obstacles we hit and how we solved them

**1. MobileNetV3's `HardSigmoid` is unsupported on MyriadX.**
MobileNetV3 uses `HardSigmoid` and `HardSwish` activations throughout its
Squeeze-and-Excite blocks. The MyriadX VPU compiler does not implement
`HardSigmoid`. We solved this by replacing all `nn.Hardsigmoid` and
`nn.Hardswish` modules with mathematically equivalent implementations
using `ReLU6`, which the Myriad does support:
- `HardSigmoid(x) = ReLU6(x + 3) / 6`
- `HardSwish(x) = x × ReLU6(x + 3) / 6`

This replacement happens before ONNX export so the unsupported ops never
appear in the graph.

**2. PyTorch's new ONNX exporter ignores `opset_version < 18`.**
PyTorch 2.6+ defaults to a new dynamo-based exporter that silently overrides
low opset versions. We set `dynamo=False` to force the legacy exporter, which
respects `opset_version=12`. Opset 12 is needed because OpenVINO 2022.x only
supports up to opset 15, and lower is safer for MobileNetV3's SE blocks.

**3. `do_constant_folding` must be `True`.**
With `do_constant_folding=False`, the SE blocks' global-average-pool output
shapes are left as dynamic, causing a channel-count mismatch during OpenVINO
conversion. Enabling constant folding resolves the shapes at export time.

**4. OpenVINO 2022.3 requires Python 3.7–3.10; Colab runs 3.12.**
We install Miniforge and create a conda environment with Python 3.10 to run
the Model Optimizer (`openvino.tools.mo`). Only this step runs in the conda
env; everything else uses Colab's default Python.

**5. `compile_tool` requires a physical Myriad device.**
OpenVINO's `compile_tool` compiles IR → blob but needs a Myriad USB device
connected. Colab has no such device. We use Luxonis's **blobconverter** cloud
API instead — it has Myriad hardware on their servers. We send the IR files
(not the ONNX) to blobconverter, which skips the Model Optimizer entirely
and only runs the Myriad compile step.

**6. `blobconverter` uses OpenVINO 2022.1, which also doesn't support `HardSigmoid`.**
This is why we can't just send the raw ONNX to blobconverter (our original
approach). By converting ONNX → IR locally with OpenVINO 2022.3 first, the
`HardSigmoid` → `ReLU6` decomposition is already baked into the IR. Blobconverter
only sees standard ops in the IR and compiles successfully.

### Preprocessing baked into the blob

The Model Optimizer flags embed ImageNet normalization into the model:
- `--mean_values [123.675, 116.28, 103.53]` (RGB channel means × 255)
- `--scale_values [58.395, 57.12, 57.375]` (RGB channel stds × 255)
- `--reverse_input_channels` (camera delivers BGR; model expects RGB)
- `-ip U8` (input precision is unsigned 8-bit, i.e., raw pixel values 0–255)

**The DepthAI pipeline on the Pi must NOT normalize frames** — the blob handles it.


### Model input/output

- **Input:** `[1, 3, 240, 320]` — batch=1, RGB, H=240, W=320 (landscape 4:3)
- **Output:** single float in `[0, 1]` — scenic quality score (sigmoid baked in)

### Dimension ordering: `(H, W)` not `(W, H)`

PyTorch's `transforms.Resize()` and `torch.randn()` use `(H, W)` ordering.
Our target frame is 320 wide × 240 tall (landscape), so the correct calls are:

- `transforms.Resize((240, 320))` — not `(320, 240)`
- `torch.randn(1, 3, 240, 320)` — not `(1, 3, 320, 240)`

The original code had these flipped, which would silently squash landscape
images into portrait orientation during training and export. This also affects
the teacher model's `infer_transform` in `hiking_model_loading.ipynb`.